In [1]:
from google.colab import drive
import os
import glob
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set Paths
BASE_DIR = "/content/drive/MyDrive/Data Science Project/raw_data"
# This goes one level up so outputs don't clutter your raw data
PROJECT_ROOT = os.path.dirname(BASE_DIR)

OUT_DIR = os.path.join(PROJECT_ROOT, "cleaned_data")
VIZ_DIR = os.path.join(PROJECT_ROOT, "visualizations")

# Create the output folders if they don't exist yet
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)

print(f"\n📁 Raw Data Directory: {BASE_DIR}")
print(f"📁 Outputs will save to: {PROJECT_ROOT}")

Mounted at /content/drive

📁 Raw Data Directory: /content/drive/MyDrive/Data Science Project/raw_data
📁 Outputs will save to: /content/drive/MyDrive/Data Science Project


In [3]:
# 3. Locate and Tally Files (Recursive search handles both flat and subfolder structures)
print("\nLocating monthly files...")
file_pattern = os.path.join(BASE_DIR, "**", "*.csv")
all_files = glob.glob(file_pattern, recursive=True)

EXPECTED_TOTAL = 60
print(f"Total files found: {len(all_files)} / {EXPECTED_TOTAL}")

if len(all_files) == 0:
    print("❌ ERROR: No files found. Please check the spelling of your Drive folders.")
elif len(all_files) != EXPECTED_TOTAL:
    print(f"⚠️ WARNING: Found {len(all_files)} files, but expected {EXPECTED_TOTAL}.")
else:
    print("✅ PERFECT MATCH: All 60 monthly files accounted for.")


Locating monthly files...
Total files found: 60 / 60
✅ PERFECT MATCH: All 60 monthly files accounted for.


In [4]:
# 4. Load Files
print("\nLoading files into Colab RAM...")
df_list = []

for file in all_files:
    try:
        monthly_df = pd.read_csv(file)
        df_list.append(monthly_df)
    except Exception as e:
        print(f"Error reading {os.path.basename(file)}: {e}")


Loading files into Colab RAM...


In [12]:
# 5. Combine and Save as CSV

print("\nCombining all months into a single dataset...")
flights_df = pd.concat(df_list, ignore_index=True)
print(f"Total rows in combined dataset: {len(flights_df):,}")

backup_path = os.path.join(PROJECT_ROOT, "nyc_flights_raw_master_2021_2025.csv")
print(f"\nExporting raw master to: {backup_path}")
print("Please wait... saving ~35.9 million rows to CSV can take 5 to 15 minutes...")

flights_df.to_csv(backup_path, index=False)

print("✅ Raw master CSV backup successfully saved to Google Drive!")


Combining all months into a single dataset...
Total rows in combined dataset: 35,887,876

Exporting raw master to: /content/drive/MyDrive/Data Science Project/nyc_flights_raw_master_2021_2025.csv
Please wait... saving ~35.9 million rows to CSV can take 5 to 15 minutes...
✅ Raw master CSV backup successfully saved to Google Drive!


In [5]:
import pandas as pd
import os

# ==========================================
# SANITY CHECK 1: Layout Alignment
# ==========================================
print("--- Checking Layout Alignment Across 60 Files ---")
reference_columns = None
mismatched_files = []

for file in all_files:
    # Read only the header row (nrows=0) to save memory and time
    cols = list(pd.read_csv(file, nrows=0).columns)

    if reference_columns is None:
        reference_columns = cols  # Set the first file as the baseline
    elif cols != reference_columns:
        mismatched_files.append((os.path.basename(file), len(cols)))

if not mismatched_files:
    print("✅ SUCCESS: All 60 files have the exact same column layout.")
    print(f"Total Columns: {len(reference_columns)}")
else:
    print(f"⚠️ WARNING: Found {len(mismatched_files)} files with a different layout.")
    for file, col_count in mismatched_files:
        print(f" - {file} has {col_count} columns (Expected {len(reference_columns)})")

--- Checking Layout Alignment Across 60 Files ---
✅ SUCCESS: All 60 files have the exact same column layout.
Total Columns: 17


In [9]:
# ==========================================
# SANITY CHECK 2: Completeness Check
# ==========================================
print("--- Checking Data Completeness (Years & Months) ---")
print("Converting FL_DATE to datetime (this may take 1-2 minutes for 35.9M rows)...")

# Convert date column to actual datetime objects
flights_df['FL_DATE'] = pd.to_datetime(flights_df['FL_DATE'])

# Extract Year and Month into temporary columns for counting
flights_df['TEMP_YEAR'] = flights_df['FL_DATE'].dt.year
flights_df['TEMP_MONTH'] = flights_df['FL_DATE'].dt.month

# Create a pivot table to count flights per month, per year
monthly_tally = pd.crosstab(
    index=flights_df['TEMP_YEAR'],
    columns=flights_df['TEMP_MONTH'],
    margins=True,
    margins_name="Total Flights"
)

# Clean up the temporary columns to keep the dataframe pristine
flights_df.drop(columns=['TEMP_YEAR', 'TEMP_MONTH'], inplace=True)

print("\n✅ SUCCESS: Monthly Flight Counts Breakdown:")
print("-" * 70)
print(monthly_tally)
print("-" * 70)
print("\nLook at the grid above. If there are hundreds of thousands of flights in every single month from 2021 to 2025, your dataset is 100% complete!")

--- Checking Data Completeness (Years & Months) ---
Converting FL_DATE to datetime (this may take 1-2 minutes for 35.9M rows)...


NameError: name 'flights_df' is not defined